In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Math et visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Utilitaires
import yaml
import json
from datetime import datetime


In [ ]:
# Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Configuration des chemins
project_path = '/content/drive/MyDrive/driver-distraction-detection'
src_path = os.path.join(project_path, 'src')

# Ajout au path système
sys.path.append(src_path)

# Import des modules personnalisés
try:
    import preprocess
    print("✓ Module preprocess importé avec succès")

except ImportError as e:
    print(f"Erreur d'importation: {e}")

✓ Module preprocess importé avec succès


In [ ]:
# Configuration des chemins
config_dir = os.path.join(project_path, 'config')
data_dir = os.path.join(project_path, 'data')
images_dir = os.path.join(data_dir, 'raw')
annotations_dir = os.path.join(data_dir, 'annotations')

# Vérification des dossiers
print("Vérification des dossiers:")
print(f"- Project: {os.path.exists(project_path)}")
print(f"- Config: {os.path.exists(config_dir)}")
print(f"- Images: {os.path.exists(images_dir)}")
print(f"- Annotations: {os.path.exists(annotations_dir)}")

Vérification des dossiers:
- Project: True
- Config: True
- Images: True
- Annotations: True


In [ ]:
# Chargement des fichiers de configuration
try:
    # Training configuration
    with open(os.path.join(config_dir, 'training.yaml'), 'r') as f:
        training_config = yaml.safe_load(f)

    # Model configuration
    with open(os.path.join(config_dir, 'model.yaml'), 'r') as f:
        model_config = yaml.safe_load(f)

    # Class mapping
    with open(os.path.join(config_dir, 'class_mapping.yaml'), 'r') as f:
        class_config = yaml.safe_load(f)

    print("✓ Toutes les configurations chargées avec succès")

except Exception as e:
    print(f"Erreur lors du chargement des configurations: {e}")

# Affichage des configurations
print("\nConfiguration d'entraînement:")
print(f"- Batch size: {training_config['dataloader']['batch_size']}")
print(f"- Époques: {training_config['training']['epochs']}")
print(f"- Learning rate: {training_config['optimizer']['lr']}")

print("\nModèles disponibles:")
for model_name in model_config['models'].keys():
    print(f"- {model_name}: {model_config['models'][model_name]['input_size']}")

print("\nClasses de détection:")
for class_id, class_info in class_config['classes'].items():
    print(f"- {class_id}: {class_info['name']}")

✓ Toutes les configurations chargées avec succès

Configuration d'entraînement:
- Batch size: 32
- Époques: 30
- Learning rate: 0.0001

Modèles disponibles:
- resnet50: (224,224)
- efficientnet_b3: (300,300)

Classes de détection:
- 0: safe_driving
- 1: texting_right
- 2: talking_phone_right
- 3: texting_left
- 4: talking_phone_left
- 5: operating_radio
- 6: drinking
- 7: reaching_behind
- 8: hair_makeup
- 9: talking_passenger


In [ ]:
# Chemins des fichiers CSV
train_csv = os.path.join(annotations_dir, 'train_labels.csv')
val_csv = os.path.join(annotations_dir, 'val_labels.csv')
test_csv = os.path.join(annotations_dir, 'test_labels.csv')

# Vérification des fichiers
print("Fichiers CSV disponibles:")
print(f"- Train: {os.path.exists(train_csv)}")
print(f"- Validation: {os.path.exists(val_csv)}")
print(f"- Test: {os.path.exists(test_csv)}")

Fichiers CSV disponibles:
- Train: True
- Validation: True
- Test: True


In [ ]:
# Chargement des données d'entraînement pour analyse
train_df = pd.read_csv(train_csv)
print(f"\nDonnées d'entraînement: {len(train_df)} images")
print(f"Colonnes: {train_df.columns.tolist()}")

# Analyse de la distribution des classes
class_distribution = train_df['label'].value_counts().sort_index()
print("\nDistribution des classes (entraînement):")

# Création d'un mapping des noms complets
class_mapping_full = {}
for class_label_str in class_distribution.index:
    # Extraction de l'ID numérique du format 'c0_safe_driving'
    try:
        numeric_class_id = int(class_label_str.split('_')[0][1:])
        class_name = class_config['classes'][str(numeric_class_id)]['name']
    except:
        # Si le format est différent, utiliser directement
        class_name = class_label_str
        numeric_class_id = 0

    class_mapping_full[class_label_str] = {
        'numeric_id': numeric_class_id,
        'name': class_name
    }

    count = class_distribution[class_label_str]
    percentage = (count / len(train_df)) * 100
    print(f"  {class_label_str}: {count:4d} images ({percentage:.1f}%)")


Données d'entraînement: 17939 images
Colonnes: ['image_path', 'label', 'label_id']

Distribution des classes (entraînement):
  c0_safe_driving: 1991 images (11.1%)
  c1_texting_right: 1814 images (10.1%)
  c2_talking_phone_right: 1853 images (10.3%)
  c3_texting_left: 1877 images (10.5%)
  c4_talking_phone_left: 1861 images (10.4%)
  c5_operating_radio: 1849 images (10.3%)
  c6_drinking: 1860 images (10.4%)
  c7_reaching_behind: 1602 images (8.9%)
  c8_hair_makeup: 1529 images (8.5%)
  c9_talking_passenger: 1703 images (9.5%)


In [ ]:
try:
    # Ajout du chemin pour importer train_classifier
    sys.path.append('/content/drive/MyDrive/driver-distraction-detection/src')

    from train_classifier import (
        load_training_config,
        setup_environment,
        build_model,
        build_loss_function,
        build_optimizer,
        build_scheduler,
        train_one_epoch,
        validate_one_epoch,
        save_checkpoint,
        load_best_model
    )
    print("✓ Fonctions d'entraînement importées avec succès")

except ImportError as e:
    print(f"Erreur d'importation des fonctions d'entraînement: {e}")
    print("Assurez-vous que train_classifier.py existe dans le dossier src")

✓ Fonctions d'entraînement importées avec succès


In [ ]:
# Choix du modèle
MODEL_NAME = "resnet50"  # Peut être changé en "efficientnet_b3"
print(f"Modèle sélectionné: {MODEL_NAME}")

# Récupération de la configuration du modèle
model_spec = model_config['models'][MODEL_NAME]
input_size = model_spec['input_size']

# Correction: s'assurer que input_size est un tuple de 2 éléments
if isinstance(input_size, list):
    input_size = tuple(input_size)
elif isinstance(input_size, int):
    input_size = (input_size, input_size)

print(f"\nConfiguration du modèle:")
print(f"- Taille d'entrée: {input_size}")
print(f"- Nombre de classes: {model_spec['num_classes']}")
print(f"- Pré-entraîné: {model_spec['pretrained']}")

Modèle sélectionné: resnet50

Configuration du modèle:
- Taille d'entrée: (224,224)
- Nombre de classes: 10
- Pré-entraîné: True


In [ ]:
# Création des transformations
print("\nCréation des transformations...")

# Vérification que input_size est correct
print(f"Taille d'entrée utilisée: {input_size}")

# Création des transformations avec la taille corrigée
try:
    train_transform = preprocess.get_train_transforms(input_size)
    val_transform = preprocess.get_val_transforms(input_size)
    test_transform = preprocess.get_test_transforms(input_size)
    print("✓ Transformations créées avec succès")
except Exception as e:
    print(f"Erreur lors de la création des transformations: {e}")
    # Fallback: utiliser une taille par défaut
    input_size = (224, 224)
    print(f"Utilisation de la taille par défaut: {input_size}")
    train_transform = preprocess.get_train_transforms(input_size)
    val_transform = preprocess.get_val_transforms(input_size)
    test_transform = preprocess.get_test_transforms(input_size)

# Paramètres des DataLoaders
batch_size = training_config['dataloader']['batch_size']
num_workers = training_config['dataloader']['num_workers']

print(f"\nParamètres DataLoader:")
print(f"- Batch size: {batch_size}")
print(f"- Workers: {num_workers}")


Création des transformations...
Taille d'entrée utilisée: (224,224)
Erreur lors de la création des transformations: If size is a sequence, it should have 1 or 2 values
Utilisation de la taille par défaut: (224, 224)

Paramètres DataLoader:
- Batch size: 32
- Workers: 4


In [ ]:
# Création des DataLoaders
print("\nCréation des DataLoaders...")

try:
    train_loader = preprocess.create_dataloader(
        csv_file=train_csv,
        images_dir=images_dir,
        transform=train_transform,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )

    val_loader = preprocess.create_dataloader(
        csv_file=val_csv,
        images_dir=images_dir,
        transform=val_transform,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    test_loader = preprocess.create_dataloader(
        csv_file=test_csv,
        images_dir=images_dir,
        transform=test_transform,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    print(f"✓ DataLoaders créés avec succès")
    print(f"- Train batches: {len(train_loader)}")
    print(f"- Validation batches: {len(val_loader)}")
    print(f"- Test batches: {len(test_loader)}")

except Exception as e:
    print(f"Erreur lors de la création des DataLoaders: {e}")



Création des DataLoaders...
✓ DataLoaders créés avec succès
- Train batches: 561
- Validation batches: 71
- Test batches: 71


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Appareil utilisé: cpu


In [ ]:
# Chargement de la configuration complète
full_config = load_training_config(
    training_cfg_path=os.path.join(config_dir, 'training.yaml'),
    model_cfg_path=os.path.join(config_dir, 'model.yaml')
)

# Création de la config adaptée pour build_model
model_build_config = {
    MODEL_NAME: model_spec,  # Ajouter la config du modèle
    **full_config  # Copier toutes les autres configurations
}

# %%
# Construction du modèle
print(f"\nConstruction du modèle {MODEL_NAME}...")

model = build_model(model_build_config, MODEL_NAME)
model = model.to(device)

# Calcul des paramètres
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Modèle construit avec succès")
print(f"- Paramètres totaux: {total_params:,}")
print(f"- Paramètres entraînables: {trainable_params:,}")


Construction du modèle resnet50...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 108MB/s]


✓ Modèle construit avec succès
- Paramètres totaux: 23,528,522
- Paramètres entraînables: 23,518,986


In [ ]:
criterion = build_loss_function(model_build_config)
optimizer = build_optimizer(model, model_build_config)
scheduler = build_scheduler(optimizer, model_build_config)

print(f"\nConfiguration d'optimisation:")
print(f"- Optimizer: {training_config['optimizer']['name']}")
print(f"- Learning rate: {training_config['optimizer']['lr']}")


Configuration d'optimisation:
- Optimizer: adamw
- Learning rate: 0.0001


In [ ]:
# ## 8. Entraînement du Modèle

# %%
# Configuration de l'entraînement
num_epochs = training_config['training']['epochs']
early_stopping_patience = training_config['training']['early_stopping_patience']

# Création du répertoire pour les checkpoints
checkpoints_dir = os.path.join(project_path, 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)

print(f"\nConfiguration d'entraînement:")
print(f"- Époques: {num_epochs}")
print(f"- Early stopping patience: {early_stopping_patience}")


Configuration d'entraînement:
- Époques: 30
- Early stopping patience: 7


In [ ]:
# Initialisation des métriques
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_epoch = 0
early_stopping_counter = 0

print(f"\n{'='*60}")
print(f"DÉBUT DE L'ENTRAÎNEMENT")
print(f"{'='*60}")

# %%
# Boucle d'entraînement
for epoch in range(num_epochs):
    print(f"\nÉpoque [{epoch+1}/{num_epochs}]")
    print("-" * 40)

    # Entraînement
    train_loss, train_acc = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    # Validation
    val_loss, val_acc = validate_one_epoch(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=device
    )

    # Mise à jour du scheduler
    if scheduler is not None:
        scheduler.step(val_loss)

    # Enregistrement des métriques
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Affichage des résultats
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    # Vérification du meilleur modèle
    is_best = val_acc > best_val_acc

    if is_best:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        early_stopping_counter = 0
        print(f"✓ Nouveau meilleur modèle! Accuracy: {val_acc:.4f}")
    else:
        early_stopping_counter += 1
        print(f"Early stopping counter: {early_stopping_counter}/{early_stopping_patience}")

    # Sauvegarde du checkpoint
    checkpoint_state = {
        'epoch': epoch + 1,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_acc': best_val_acc,
        'config': model_build_config
    }

    save_checkpoint(
        state=checkpoint_state,
        config=model_build_config,
        is_best=is_best
    )

    # Early stopping
    if early_stopping_counter >= early_stopping_patience:
        print(f"\n Early stopping déclenché à l'époque {epoch+1}")
        break

print(f"\n{'='*60}")
print("ENTRAÎNEMENT TERMINÉ")
print(f"{'='*60}")


In [ ]:
#------------------------hadchi li ltht à tester après le training ----------------------------
#------------------------training kidi lw9t ---------------------------

In [ ]:
# ## 9. Sauvegarde du Modèle (IMMÉDIATEMENT après l'entraînement)

# %%
# Sauvegarde du modèle avant toute évaluation
model_save_path = os.path.join(project_path, f'{MODEL_NAME}_trained.pth')

torch.save({
    'epoch': best_epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
    'best_val_acc': best_val_acc,
    'training_history': history,
    'config': model_build_config,
    'model_name': MODEL_NAME,
    'input_size': input_size,
    'num_classes': model_spec['num_classes']
}, model_save_path)

print(f"✓ Modèle sauvegardé IMMÉDIATEMENT après entraînement: {model_save_path}")
print(f"  - Meilleure accuracy: {best_val_acc:.4f}")
print(f"  - Meilleure époque: {best_epoch}")


In [ ]:
# ## 10. Évaluation avec evaluate.py

# %%
# Import des fonctions d'évaluation
from evaluate import evaluate_models

# %%
# Prédictions sur l'ensemble de validation
val_predictions = {}
val_true_labels = []

model.eval()  # Mode évaluation
with torch.no_grad():
    val_preds = []
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        val_preds.extend(preds.cpu().numpy())
        val_true_labels.extend(labels.numpy())

val_predictions[MODEL_NAME] = np.array(val_preds)
val_true_labels = np.array(val_true_labels)

In [ ]:
# Évaluation sur validation
class_names = [class_config['classes'][str(i)]['name'] for i in range(10)]

results_val = evaluate_models(
    y_true=val_true_labels,
    predictions_dict=val_predictions,
    class_names=class_names,
    save_results=True,
    results_dir=os.path.join(project_path, 'evaluation_results')
)


In [ ]:
# Prédictions sur l'ensemble de test
test_predictions = {}
test_true_labels = []

model.eval()
with torch.no_grad():
    test_preds = []
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().numpy())
        test_true_labels.extend(labels.numpy())

test_predictions[MODEL_NAME] = np.array(test_preds)
test_true_labels = np.array(test_true_labels)

# Évaluation sur test
results_test = evaluate_models(
    y_true=test_true_labels,
    predictions_dict=test_predictions,
    class_names=class_names,
    save_results=True,
    results_dir=os.path.join(project_path, 'test_results')
)


In [ ]:
# ## 12. Sauvegarde Finale (optionnelle)

# %%
# Sauvegarde additionnelle avec les résultats d'évaluation
final_save_path = os.path.join(project_path, f'{MODEL_NAME}_final.pth')

torch.save({
    'model_state_dict': model.state_dict(),
    'model_name': MODEL_NAME,
    'input_size': input_size,
    'num_classes': model_spec['num_classes'],
    'class_names': class_names,
    'validation_accuracy': float(results_val['metrics'][MODEL_NAME]['accuracy']),
    'test_accuracy': float(results_test['metrics'][MODEL_NAME]['accuracy']),
    'training_info': {
        'best_epoch': best_epoch,
        'best_accuracy': float(best_val_acc),
        'total_epochs': len(history['train_loss'])
    }
}, final_save_path)

print(f"\n✓ Modèle final avec métriques sauvegardé: {final_save_path}")